# Query Transformations

Rewrite, step-back and decompose the question before retrieval.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def rewrite(q):
    return llm(f"Rewrite this as a clear, standalone search query. Reply with the query only.\n\n{q}")

def step_back(q):
    return llm(f"Write a more general question that would retrieve useful background for this one. Reply with the question only.\n\n{q}")

def decompose(q):
    return llm_json(f"Break this question into 2-3 simple sub-questions. Reply with a JSON list of strings.\n\n{q}") or [q]

original = "so how do u make the search thing better at finding exact stuff like product codes and also fuzzy meaning?"
print("rewrite  :", rewrite(original))
print("step-back:", step_back(original))
print("sub-qs   :", decompose(original))

Retrieve with **every** variant and merge the results, keeping the best score per chunk.

In [ ]:
def multi_query_search(q, k=4):
    queries = [q, rewrite(q), step_back(q)] + decompose(q)
    best = {}
    for sub in queries:
        for h in store.search(sub, k):
            if h["idx"] not in best or h["score"] > best[h["idx"]]["score"]:
                best[h["idx"]] = h
    return sorted(best.values(), key=lambda h: -h["score"])[:k]

show(multi_query_search(original))
print(answer(original, [h["text"] for h in multi_query_search(original)]))